Tester

In [19]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_rows = 100

df_clean = pd.DataFrame(
    {
        "feature_a": np.random.normal(loc=0.0, scale=1.0, size=n_rows),
        "feature_b": np.random.uniform(low=0, high=10, size=n_rows),
        "feature_c": np.random.poisson(lam=5, size=n_rows),
        "feature_d": np.random.normal(loc=10.0, scale=2.0, size=n_rows),
        "feature_e": np.random.binomial(n=1, p=0.3, size=n_rows),
    }
)

df_clean.head()

,feature_a,feature_b,feature_c,feature_d,feature_e
0,0.496714,4.174110,6,6.994470,0
1,-0.138264,2.221078,2,9.478530,1
2,0.647689,1.198654,4,7.749162,0
3,1.523030,3.376152,9,9.701927,0
4,-0.234153,9.429097,7,9.986693,0


In [20]:
n_error_rows = 20

# Sample rows from the clean dataframe
df_error = df_clean.sample(n_error_rows, random_state=1).reset_index(drop=True)

# --- Inject errors ---

# Additive constant error
df_error["feature_a"] = df_error["feature_a"] + 2.5

# Scaling error
df_error["feature_b"] = df_error["feature_b"] * 1.8

# Increased noise
df_error["feature_c"] = df_error["feature_c"] + np.random.normal(loc=0, scale=2.0, size=n_error_rows)

# Missing values (randomly drop ~30%)
mask_missing = np.random.rand(n_error_rows) < 0.3
df_error.loc[mask_missing, "feature_d"] = np.nan

# Logical / data-entry error: flip binary with noise
flip_mask = np.random.rand(n_error_rows) < 0.2
df_error.loc[flip_mask, "feature_e"] = 1 - df_error.loc[flip_mask, "feature_e"]

df_error.head()

,feature_a,feature_b,feature_c,feature_d,feature_e
0,2.280328,9.877208,1.254739,NaN,0
1,1.691506,12.819226,7.587316,9.061705,0
2,1.442289,9.643944,3.399279,11.055530,0
3,2.857113,12.454114,1.390426,NaN,0
4,2.172338,6.618884,4.987768,5.260557,0


In [21]:
# cleaner (uses inference)
import pandas as pd

from conformal_data_cleaning.cleaner.autogluon import ConformalAutoGluonCleaner

# Make cleaner
cleaner: ConformalAutoGluonCleaner = ConformalAutoGluonCleaner(confidence_level= 0.999, seed = 42)
fit_cleaner = cleaner.fit(df_clean)
cleaned_data: tuple[pd.DataFrame, pd.DataFrame] = fit_cleaner.transform(df_error)
data, mask = cleaned_data
print((data != df_error).sum())
print(mask.sum())

2025-12-12 12:09:15,897 - INFO - conformal_data_cleaning.cleaner.autogluon: Start fitting predictor #1 of 5
2025-12-12 12:09:15,897 - INFO - conformal_data_cleaning.cleaner.autogluon: Start fitting predictor #1 of 5
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #88~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Oct 14 14:03:14 UTC 2
CPU Count:          8
Memory Avail:       5.97 GB / 11.46 GB (52.1%)
Disk Space Avail:   124.33 GB / 467.89 GB (26.6%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme' : New in v1.4: Massively better than 'best' on datasets <30000 samples by using new models meta-learn

feature_a    17
feature_b    13
feature_c    20
feature_d     9
feature_e     0
dtype: int64
feature_a    17
feature_b    13
feature_c    20
feature_d     9
feature_e     0
dtype: int64


Full Evaluation

In [22]:
import glob
import os

from tab_err.api.high_level import create_errors
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import train_test_split
import json


csv_files = glob.glob(os.path.join("./data", "*.csv"))
dataframes = [pd.read_csv(f) for f in csv_files]



In [23]:
for d in dataframes:
    print(d["target"].head())

0    1
1    1
2    1
3    1
4    1
Name: target, dtype: int64
0    good
1     bad
2    good
3    good
4     bad
Name: target, dtype: object
0    1
1    1
2    1
3    1
4    1
Name: target, dtype: int64
0      UP
1      UP
2      UP
3      UP
4    DOWN
Name: target, dtype: object
0    tested_positive
1    tested_negative
2    tested_positive
3    tested_negative
4    tested_positive
Name: target, dtype: object
0    Z
1    P
2    S
3    H
4    H
Name: target, dtype: object
0     van
1     van
2    saab
3     van
4     bus
Name: target, dtype: object


In [24]:
def detect_task(y: pd.Series) -> str:
    """Return 'regression' or 'classification'."""
    # Simple heuristic: numeric with many unique values → regression
    if pd.api.types.is_numeric_dtype(y):
        if y.nunique() > 20:
            return "regression"
        else:
            return "classification"
    else:
        return "classification"

def get_model(task: str):
    return RandomForestRegressor() if task == "regression" else RandomForestClassifier()


def evaluate_model(model, X_test, y_test, task):
    # Predict on test data
    preds = model.predict(X_test)

    if task == "classification":
        acc = accuracy_score(y_test, preds)
        return 1 - acc   # classification error
    else:
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        return rmse      # regression error


def process_dataset(df, cleaner_cls):
    """Run the 3-step pipeline on a single dataframe."""
    df = df.dropna(subset=["target"]).copy()

    # --- 1. Holdout split ---
    clean_df, test_df = train_test_split(df, test_size=0.5, random_state=42)


    X_train = clean_df.drop(columns=["target"])
    y_train = clean_df["target"]


    # Automatically detect regression vs classification
    task = "classification"  #detect_task(y_train)
    model = get_model(task)
    model.fit(X_train, y_train)

    X_test = test_df.drop(columns=["target"])
    y_test = test_df["target"]



    # -------------------------
    # A) Baseline model
    # -------------------------
    baseline_err = evaluate_model(model, X_test, y_test, task)

    # -------------------------
    # B) Error features model
    # -------------------------

    # Tab err HLA
    X_test_errored, _ = create_errors(X_test, error_rate=0.5)

    error_features_err = evaluate_model(model, X_test_errored, y_test, task)

    # -------------------------
    # C) Cleaned model
    # -------------------------
    cleaner = cleaner_cls(confidence_level=0.999, seed=42)

    # Fit cleaner on holdout data (df_error)
    fit_cleaner = cleaner.fit(X_train)
    X_test_cleaned, _ = fit_cleaner.transform(X_test_errored)


    cleaned_err = evaluate_model(model, X_test_cleaned, y_test, task)

    return {
        "baseline_error": baseline_err,
        "error_features_error": error_features_err,
        "cleaned_error": cleaned_err,
        "task": task,
    }

def evaluate_all(dfs, cleaner_cls):
    results = []
    for i, df in enumerate(dfs):
        print(f"Processing dataset {i+1}/{len(dfs)} ...")
        res = process_dataset(df, cleaner_cls)
        results.append(res)
        with open(f"{i}.json", "w") as f:
            json.dump(res, f, indent=2)

    return pd.DataFrame(results)

In [25]:
results = evaluate_all(dataframes, ConformalAutoGluonCleaner)
print(results)

Processing dataset 1/7 ...


/home/chandlernick/BHT/Research/conformal-data-cleaning/.venv/lib/python3.12/site-packages/tab_err/error_type/_error_type.py:67: UserWarning: self.config.add_delta_value is none, sampling a random delta value uniformly from the range of column: word_freq_make.
  return self._apply(data, error_mask, column)
/home/chandlernick/BHT/Research/conformal-data-cleaning/.venv/lib/python3.12/site-packages/tab_err/error_type/_error_type.py:67: UserWarning: No scaling function was supplied for WrongUnit, defaulting to multiplication by 10.0.
  return self._apply(data, error_mask, column)
2025-12-12 12:09:29,639 - INFO - conformal_data_cleaning.cleaner.autogluon: Start fitting predictor #1 of 57
2025-12-12 12:09:29,639 - INFO - conformal_data_cleaning.cleaner.autogluon: Start fitting predictor #1 of 57
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.11
Operating System:   Linux
Platform Machine:   x86_64
Platform

KeyboardInterrupt: 